# T19 — Time-Travel Snapshots

Generate monthly point-in-time snapshots with configurable growth, churn, and seasonality.

**What you'll learn:**
- Generate 12 months of evolving data
- Configure growth rates and seasonal spikes
- Produce partitioned DataFrames with `_snapshot_date`
- Write partitioned Parquet for Lakehouse queries

## 1. Generate 12-Month Snapshots

In [ ]:
from sqllocks_spindle import RetailDomain, TimeTravelEngine, TimeTravelConfig

engine = TimeTravelEngine()
result = engine.generate(
    domain=RetailDomain(),
    config=TimeTravelConfig(
        months=12,
        start_date="2025-01-01",
        growth_rate=0.05,           # 5% monthly growth
        churn_rate=0.02,            # 2% monthly churn
        seasonality={11: 1.5, 12: 2.0},  # Holiday spike
        seed=42,
    ),
    scale="small",
)

print(result.summary())

## 2. Inspect Snapshots

In [ ]:
for snap in result.snapshots:
    customer_count = snap.row_counts.get('customer', 0)
    order_count = snap.row_counts.get('order', 0)
    print(f"Month {snap.month_index:2d} ({snap.snapshot_date}): "
          f"{customer_count:,} customers, {order_count:,} orders")

## 3. Growth Visualization

In [ ]:
months = [s.month_index for s in result.snapshots]
customers = [s.row_counts.get('customer', 0) for s in result.snapshots]

print("Customer Growth Over 12 Months:")
for m, c in zip(months, customers):
    bar = '#' * (c // 20)
    print(f"  Month {m:2d}: {c:5,} {bar}")

print(f"\nNet growth: {customers[0]:,} -> {customers[-1]:,} "
      f"({(customers[-1]/customers[0] - 1)*100:.1f}%)")

## 4. Partitioned Output

In [ ]:
partitioned = result.to_partitioned_dfs()

for table_name, df in partitioned.items():
    dates = df['_snapshot_date'].nunique()
    print(f"{table_name}: {len(df):,} total rows across {dates} snapshots")

print("\nSample with _snapshot_date column:")
partitioned['customer'][['customer_id', '_snapshot_date']].head(10)

## 5. Write to Parquet (Lakehouse Ready)

```python
for table_name, df in partitioned.items():
    df.to_parquet(f"/lakehouse/Files/snapshots/{table_name}.parquet")
```

## 6. CLI Equivalent

```bash
spindle time-travel retail --months 12 --output ./snapshots/ \
  --growth-rate 0.05 --churn-rate 0.02 --format parquet
```